<a href="https://colab.research.google.com/github/oluwasegunjacobsoj/3mttxNIPOST/blob/main/3mttxNIPOST_modeling_based.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
# Option A - Set it directly in code for testing only, never commit this
os.environ["POSTCODE_API_KEY"] = "nipost_test_47526af9fd9c6346ad442a91da95ee6bf47e760a2f0c6b8e"

# Option B - Check what Python sees
print("Key loaded?", os.getenv("POSTCODE_API_KEY") is not None)
print("First 15 chars:", str(os.getenv("POSTCODE_API_KEY"))[:15])
print("Key length:", len(str(os.getenv("POSTCODE_API_KEY"))))

Key loaded? True
First 15 chars: nipost_test_475
Key length: 60


In [ ]:
import requests
API_KEY = "nipost_test_4bc90594d90abf3f1848c2e46c05391736215ddefaa59cc7"
print("Key len:", len(API_KEY))

resp = requests.get(
    "https://api.postcode.gov.ng/v1/lookup?code=FC-01-A01-KP-27&level=1",
    headers={"X-API-Key": API_KEY}
)
print("Status:", resp.status_code)
print(resp.text)

Key len: 60
Status: 200
{"data":{"postcode":"FC-01-A01-KP-27","valid":true,"status":"valid","verified":false}}



In [ ]:
import requests
import pandas as pd
import numpy as np

API_KEY = "nipost_test_4bc90594d90abf3f1848c2e46c05391736215ddefaa59cc7"  # put your new key
BASE_URL = "https://api.postcode.gov.ng"
HEADERS = {"X-API-Key": API_KEY}

REAL_POSTCODES = [
    "FC-01-A01-KP-27",
    "FC-01-A01-LR-01",
    "FC-01-A01-MH-01",
    "FC-01-A01-MV-01",
    "FC-01-A01-MW-01",
]

postcode_rows = []
for code in REAL_POSTCODES:
    resp = requests.get(f"{BASE_URL}/v1/lookup?code={code}&level=1", headers=HEADERS)
    data = resp.json()["data"]
    if data.get("valid"):
        parts = code.split("-")
        postcode_rows.append({
            "postcode": code, "state_code": parts[0], "lga_code": parts[1],
            "district_code": parts[2], "area_code": parts[3], "unit_code": parts[4]
        })

postcode_df = pd.DataFrame(postcode_rows)
postcode_df.to_csv("postcode_dimension.csv", index=False)
print(f"Validated {len(postcode_df)} postcodes")

# Generate 300 customers per postcode = 1500 customers
np.random.seed(42)
operators = ["MTN", "Airtel", "Glo", "9mobile"]
rows = []
cust_id = 0
for _, pc in postcode_df.iterrows():
    for _ in range(300):
        cust_id += 1
        signal = np.random.normal(-85, 12)
        days_no_network = max(0, int(np.random.poisson(2) if signal < -95 else np.random.poisson(0.5)))
        recharge_drop = np.random.normal(10, 15) if signal < -95 else np.random.normal(0, 8)
        complaints = np.random.poisson(2) if signal < -95 else np.random.poisson(0.3)
        tenure = np.random.randint(1, 60)
        churn_prob = 0.05 + 0.02*days_no_network + 0.01*max(0, recharge_drop) + 0.08*complaints - 0.002*tenure
        churned = 1 if np.random.rand() < np.clip(churn_prob, 0, 0.9) else 0
        rows.append([f"CUST_{cust_id}", pc["postcode"], pc["state_code"], pc["lga_code"],
                     tenure, np.random.randint(500, 10000), max(-5, recharge_drop),
                     np.random.uniform(0.5, 20), np.random.uniform(0.01, 0.3),
                     signal, days_no_network, complaints,
                     np.random.choice(operators),
                     1 if signal < -100 else 0,
                     np.random.uniform(0.05, 0.4), churned])

cols = ["customer_id","postcode","state_code","lga_code","tenure_months",
"avg_monthly_recharge_naira","recharge_drop_pct_last_30d","data_usage_gb_last_30d",
"call_drop_rate","avg_signal_strength_dbm","days_with_no_network_last_30d",
"complaints_last_90d","network_operator","is_postcode_blackspot",
"neighbour_churn_rate_postcode","churned_in_30_days"]

df = pd.DataFrame(rows, columns=cols)
df.to_csv("telco_churn_postcode_real.csv", index=False)
print(f"Generated {len(df)} customers")
print(df.head())

Validated 5 postcodes
Generated 1500 customers
  customer_id         postcode state_code lga_code  tenure_months  \
0      CUST_1  FC-01-A01-KP-27         FC       01             19   
1      CUST_2  FC-01-A01-KP-27         FC       01             58   
2      CUST_3  FC-01-A01-KP-27         FC       01             37   
3      CUST_4  FC-01-A01-KP-27         FC       01             17   
4      CUST_5  FC-01-A01-KP-27         FC       01             57   

   avg_monthly_recharge_naira  recharge_drop_pct_last_30d  \
0                        6078                   -1.106114   
1                        9166                   -1.276132   
2                        5893                   -1.310937   
3                        4443                   -5.000000   
4                        8392                    7.054088   

   data_usage_gb_last_30d  call_drop_rate  avg_signal_strength_dbm  \
0               17.390435        0.184323               -79.039430   
1               12.431131      

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

df = pd.read_csv("telco_churn_postcode_real.csv")
X = df.drop(columns=["customer_id", "churned_in_30_days"])
y = df["churned_in_30_days"]

cat_cols = ["postcode", "state_code", "lga_code", "network_operator"]
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

# Try XGBoost, fallback to RandomForest if not installed
try:
    import xgboost as xgb
    clf = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric="logloss")
    print("Using XGBoost")
except ImportError:
    from sklearn.ensemble import RandomForestClassifier
    clf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
    print("Using RandomForest - XGBoost not installed")

model = Pipeline([("pre", pre), ("clf", clf)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, pred))
print("AUC:", round(roc_auc_score(y_test, proba), 3))
print("Overall churn rate:", round(y.mean(), 3))

# Presentation table
df["churn_prob"] = model.predict_proba(X)[:, 1]
summary = df.groupby(["postcode", "network_operator"]).agg(
    total_customers=("customer_id", "count"),
    predicted_churn=("churned_in_30_days", "sum"),
    churn_rate=("churned_in_30_days", "mean"),
    avg_signal=("avg_signal_strength_dbm", "mean"),
    avg_days_no_network=("days_with_no_network_last_30d", "mean")
).reset_index().sort_values("churn_rate", ascending=False)
print(summary.to_string())
summary.to_csv("churn_by_postcode_operator.csv", index=False)

Using XGBoost
              precision    recall  f1-score   support

           0       0.92      0.98      0.95       268
           1       0.64      0.28      0.39        32

    accuracy                           0.91       300
   macro avg       0.78      0.63      0.67       300
weighted avg       0.89      0.91      0.89       300

AUC: 0.806
Overall churn rate: 0.105
           postcode network_operator  total_customers  predicted_churn  churn_rate  avg_signal  avg_days_no_network
10  FC-01-A01-MH-01              Glo               79               13    0.164557  -84.382948             0.873418
15  FC-01-A01-MV-01              MTN               76               12    0.157895  -86.849831             1.118421
19  FC-01-A01-MW-01              MTN               72               10    0.138889  -83.709033             0.888889
6   FC-01-A01-LR-01              Glo               81               11    0.135802  -84.914982             0.790123
18  FC-01-A01-MW-01              Glo      

In [ ]:
# Lower threshold from 0.5 to 0.30 to catch more churners
proba = model.predict_proba(X_test)[:, 1]
pred_030 = (proba >= 0.30).astype(int)
from sklearn.metrics import classification_report
print("Threshold 0.30:")
print(classification_report(y_test, pred_030))

Threshold 0.30:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       268
           1       0.48      0.41      0.44        32

    accuracy                           0.89       300
   macro avg       0.71      0.68      0.69       300
weighted avg       0.88      0.89      0.89       300



In [ ]:
import numpy as np
from sklearn.metrics import f1_score, recall_score, precision_score

proba = model.predict_proba(X_test)[:, 1]
for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred = (proba >= t).astype(int)
    print(f"Threshold {t}: Precision={precision_score(y_test, pred, zero_division=0):.2f} Recall={recall_score(y_test, pred):.2f} F1={f1_score(y_test, pred):.2f}")

Threshold 0.2: Precision=0.42 Recall=0.47 F1=0.44
Threshold 0.25: Precision=0.44 Recall=0.44 F1=0.44
Threshold 0.3: Precision=0.48 Recall=0.41 F1=0.44
Threshold 0.35: Precision=0.48 Recall=0.34 F1=0.40
Threshold 0.4: Precision=0.53 Recall=0.31 F1=0.39
Threshold 0.5: Precision=0.64 Recall=0.28 F1=0.39
